## See "Hard Mining Negatives for Semantic Similarity"

https://www.kaggle.com/code/jithinanievarghese/hard-mining-negatives-for-semantic-similarity#Load-Data-and-preprocess-data

In [18]:
import os
import sys
PROJECT_ROOT = os.path.abspath(os.path.join(
 os.getcwd(),
 os.pardir+'/playground')
)
#only add it once
if (PROJECT_ROOT not in sys.path):
 sys.path.append(PROJECT_ROOT)

import utils as ut
from sentence_transformers import SentenceTransformer, util
from tqdm import tqdm
import numpy as np 
import pandas as pd 
import csv

In [19]:
def preprocess_text(text):
    """
    clean white space and lower case the text
    """
    return " ".join(text.split()).lower()

In [20]:
df = pd.read_json('../data/trn.json')

df.drop_duplicates(subset=['anchor', 'positive'], inplace=True)
# df.drop_duplicates(subset=['anchor'], inplace=True)
# df.drop_duplicates(subset=['positive'], inplace=True)
df.reset_index(drop=True, inplace=True)
df.anchor = df['anchor'].apply(lambda x: preprocess_text(x))
df.positive = df['positive'].apply(lambda x: preprocess_text(x))

df.head()

,anchor,positive,most_dissimilar_context,id
0,what safeguards are in place to protect the in...,information we collect from other sources we m...,If such Standard Cost methodology change resul...,0
1,is there a guarantee from the manufacturers re...,each of the suppliers warrants that the produc...,We do not obtain your age range and gender.,1
2,what type of authorization has the video confe...,skype hereby grants to online bvi and the comp...,"In order to keep your exclusivity, you agree t...",2
3,can the blockchain administrator arrange for t...,(a) the fund hereby employs the blockchain adm...,8. Insurance. During the Term of this Agreemen...,3
4,what happens if a party fails to retain record...,each party will retain such records for at lea...,"""Web Beacons"" (also known as Web bugs, pixel t...",4


# Embed the columns of interest

### Get list of positives and anchors

In [21]:
%%time
from sentence_transformers import InputExample
from tqdm.auto import tqdm  # so we see progress bar
def getsentencelists(df,cols):
    '''
    param: df dataframe
    param: cols list of columns in dataframe to return lists from
    return: tuple of lists, each list is a string of all strings in column
     of InputExample objects
     ex.
     cols=['positive','anchor']
    positives, anchors = getsentencelists(df,cols)
    ''' 
    res={}  
    for col in cols:
        res[col]=[]
    for _,row in tqdm(df.iterrows()):
        for col in cols:
            res[col].append(row[col])
    return (res[col] for col in cols)

cols=['positive','anchor']
positives, anchors = getsentencelists(df,cols)


35258it [00:01, 34289.83it/s]

CPU times: user 1.02 s, sys: 10.1 ms, total: 1.03 s
Wall time: 1.03 s


### generate embeddings

In [5]:
%%time

import logging
import torch
import numpy as np

from sentence_transformers import LoggingHandler, SentenceTransformer

#### Just some code to print debug information to stdout
np.set_printoptions(threshold=100)

logging.basicConfig(
    format="%(asctime)s - %(message)s", datefmt="%Y-%m-%d %H:%M:%S", level=logging.INFO, handlers=[LoggingHandler()]
)
#### /print debug information to stdout


# Load pre-trained Sentence Transformer Model. It will be downloaded automatically
model = SentenceTransformer("all-MiniLM-L6-v2",device="cuda:0" if torch.cuda.is_available() else "cpu",)

# Use "convert_to_tensor=True" to keep the tensors on GPU (if available)
positive_embeddings = model.encode(positives, convert_to_tensor=True)
anchor_embeddings = model.encode(anchors, convert_to_tensor=True)


/home/kperkins411/anaconda3/envs/p311/lib/python3.11/site-packages/huggingface_hub/file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


2024-07-06 18:26:00 - Load pretrained SentenceTransformer: all-MiniLM-L6-v2


Batches: 100%|██████████| 1102/1102 [00:11<00:00, 93.51it/s] 

CPU times: user 25min 59s, sys: 43.8 s, total: 26min 43s
Wall time: 36 s


### get similarity score matrix

In [7]:
# We use cosine-similarity 
scores=model.similarity(anchor_embeddings, positive_embeddings)
# similarity_scores_matrix.shape
# scores.device


### Hard negative mine the positives for similar positives

In [12]:
import torch
from itertools import compress
from tqdm.auto import tqdm

def get_hard_negatives(scores,contexts, low=0.7, high=0.74):
    """
    Train a sentencetransformer model, get its average similarity score, use range around that average for hard
    negatives
    expects scores to be nxn matrix of similarity scores
    expects contexts to be a list of n strings
    expects low and high to be floats denoting the range of acceptable similarity scores
    Get pairs of indices with low<= score <= high
    returns: list of contexts whose similarity score is between low and high
    """
    low=torch.tensor([low])
    high=torch.tensor([high])

    if scores.is_cuda:
        low=low.to(scores.device)
        high=high.to(scores.device)

    # Get the entries between low and high
    l=torch.gt(scores,low)
    h=torch.lt(scores,high)
    selections=(l&h).tolist()

    # use selections to select hard negatives
    hard_negatives=[]
    for i,row in tqdm(enumerate(selections)):
        #do not include the current rows context
        row[i]=False

        #get all the other contexts
        res=list(compress(contexts,row))
        
        #if no context then get the max similarity score that is not the positive
        if(len(res)==0):
            #if no hard negatives, select the max similarity_score, that is not the positive 
            top,ind=torch.topk(scores[i],2)
            #the following assummes the first [0] is the similarity score of the positive
            res=[contexts[ind[1]]]
        hard_negatives.append(res)
    return hard_negatives

In [13]:
ghn=get_hard_negatives(scores,positives)

35258it [00:19, 1841.05it/s]


In [17]:
#lets see how they look
# for p in ghn[:20]: print(len(p))
print(f'ANCHOR={anchors[0]}')
print(f'POSITIVE={positives[0]}')
print(f'HARDNEG={ghn[0]}')

ANCHOR=what safeguards are in place to protect the information obtained from third-party sources?
POSITIVE=information we collect from other sources we may also receive information from other sources and combine that with information we collect through our services. for example: if you choose to link, create, or log in to your uber account with a payment provider (e.g., google wallet) or social media service (e.g., facebook), or if you engage with a separate app or website that uses our api (or whose api we use), we may receive information about you or your connections from that site or app.
HARDNEG=['2.5 neither party shall be required to keep confidential any information which is, or becomes, publicly available, is independently developed by either party outside the scope of this agreement, or is rightfully obtained from third parties.']


In [22]:
#test
scores1=model.similarity(anchor_embeddings[:5], positive_embeddings[:5])
pos=positives[:5]
anc=anchors[:5]

ghn=get_hard_negatives(scores1,pos)
print(ghn)

for p in ghn[:20]: print(len(p))
print(scores1)
for p in pos: print(p)

5it [00:00, 8253.25it/s]

[['each party will retain such records for at least three (3) years following expiration or termination of this agreement or such longer period as may be required by applicable law or regulation.'], ['each party will retain such records for at least three (3) years following expiration or termination of this agreement or such longer period as may be required by applicable law or regulation.'], ['(a) the fund hereby employs the blockchain administrator to act as the blockchain administrator of the fund, and to furnish, or arrange for others to furnish, the services, personnel and facilities described below, subject to review by and the overall control of the fund\'s board of trustees (the "board"), for the period and on the terms and conditions set forth in this agreement.'], ['skype hereby grants to online bvi and the company a limited, non-exclusive, non-sublicensable (except as set forth herein), non-transferable, non-assignable (except as provided in section 14.4), royalty-free (but

# Junk

In [7]:
class HardMineNegatives():
    """
    Hard-mining Negatives for training a semantic similairty task with Triplet Loss.
    Here we find the nearest negatives of a query in a search pool 
    by using sentence transformer model embeddings and cosine similarity ratio.
    param: model_path: path of sentence transformer model
    param: search_max_threshold:  maximimum cosine similarity ratio
    param: search_min_threshold: minimum cosine similarity ratio
    param: search_limit: total length of data in which we want to search, only if search pool length is very high
    param: top_n_results: number of top nearest negative  to be returned, default is 1
    """
    def __init__(self, model_path: str, **kwargs):
        self.model = SentenceTransformer(model_path)
        self.search_max_threshold = kwargs['search_max_threshold'],
        self.search_min_threshold = kwargs['search_min_threshold']
        self.search_limit = kwargs.get('search_limit')
        self.top_n_results = kwargs.get('top_n_results') if kwargs.get('top_n_results') else 1

    def get_hard_mined_negatives(self, anchor: str,  search_pool:np.ndarray):
        """
        to retrieve embeddings from sentence transformer model for anchor and sentences in search pool,
        find the cosine similairty ratio between the  anchor and search pool sentences,
        apply search thresholds and return the top nearest negatives based on the highest
        cosine similarity scores.
        if no data is found in between the self.search_max_threshold and self.search_min_threshold ratios,
        we will take the results between 0 and less than self.search_min_threshold ratios (this is an extreme case)

        param: anchor: source text to which we need to find the nearest negative
        param: search_pool: numpy array of sentences from which
               we need to find the cosine similarity ratios with the anchor.
               any meta value for sentences can be given after next index of
               sentence, in the form
               search_pool = array([
                    ['apple iphone 8 256 gb gold', "mobile", "1001"],
                    ['apple iphone 7 plus 32gb silver', "1002"]])
               where "mobile", "1001" are meta values,
               the returned results will contain the respective cosine similarity
               ratio at the last index of each sentence array
               result = array([
                    ['apple iphone 8 256 gb gold', "mobile", "1001", 69.5],
                    ['apple iphone 7 plus 32gb silver', "1002", 70.5]])
               where 69.5 and 70.5 are cosine similarity ratios.
        """
        self.search_limit = self.search_limit if self.search_limit else search_pool.shape[0]
        search_pool = search_pool[: self.search_limit]
        # shuffle data to search in random pool of data, in case of search limit less than the total length
        np.random.shuffle(search_pool)
        sentences = [anchor] + [row[0] for row in search_pool]
        embeddings = self.model.encode(sentences, convert_to_tensor=False)
        source_vector = embeddings[0]
        # calculate the cosine similairty with the other sentences in search pool
        similarity = [round(util.cos_sim(source_vector, embed).numpy()[0][0]*100, 2) for embed in embeddings[1:]]
        similarity = np.array(similarity)
        negative_indices = np.where((similarity <= self.search_max_threshold) & (similarity >= self.search_min_threshold))
        if not negative_indices[0].shape[0]:
            negative_indices = np.where((similarity < self.search_min_threshold) & (similarity >= 0))
        negative_indices = negative_indices[0]
        # take respective selected indices
        search_pool = np.take(search_pool, negative_indices, axis=0)
        similarity = np.take(similarity, negative_indices, axis=0)
        # reshape to concatenate with meta values of search pool
        similarity = similarity.reshape(-1, 1)
        # concat the ratio to the meta values of search pool
        search_pool = np.concatenate((search_pool, similarity), axis=1)
        # sort the data in descending order
        search_pool = search_pool[search_pool[:, -1].argsort()][::-1]
        return search_pool[:self.top_n_results]

In [8]:
model_path = 'sentence-transformers/all-MiniLM-L6-v2'
obj = HardMineNegatives(
    model_path=model_path,
    search_max_threshold=65,
    search_min_threshold=50,
    search_limit=None,
    top_n_results=1)

/home/kperkins411/anaconda3/envs/p311/lib/python3.11/site-packages/huggingface_hub/file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


In [9]:
final_results = []
for row in tqdm(df.to_dict('records')):
    anchor = row['anchor']
    search_pool = df[df.anchor != anchor]
    search_pool.reset_index(drop=True, inplace=True)
    search_pool = search_pool.drop_duplicates()
    search_pool = search_pool.loc[:, ['positive']]
    search_pool = search_pool.to_numpy()
    print(f'Mining negatives for "{anchor}"')
    top_results = obj.get_hard_mined_negatives(anchor, search_pool)
    anchor = np.array([anchor]).reshape(-1, 1)
    anchor = np.repeat(anchor, repeats=len(top_results), axis=0)
    results_with_meta = np.concatenate((anchor, top_results), axis=1)
    final_results.extend(results_with_meta.tolist())

  0%|          | 0/3347 [00:00<?, ?it/s]

Mining negatives for "what safeguards are in place to protect the information obtained from third-party sources?"


  0%|          | 1/3347 [20:58<1169:41:31, 1258.49s/it]

Mining negatives for "is there a guarantee from the manufacturers regarding the conformity of the items to the mutually approved written standards for a certain duration?"


  0%|          | 2/3347 [21:00<482:35:10, 519.38s/it]  

...

Mining negatives for "what materials has cano provided?"


  0%|          | 9/3347 [21:14<22:52:07, 24.66s/it]

Mining negatives for "can the company offset the distributor's debts against the repurchase price?"


  0%|          | 10/3347 [21:16<16:21:27, 17.65s/it]

Mining negatives for "what is the scope of 'our app' as mentioned in the clause?"


In [ ]:
# import torch

# a = torch.Tensor([[12, 1, 0, 0],
#                   [4, 9, 21, 1],
#                   [10, 2, 1, 0]])

# b = torch.rand(3, 4, 8)

# print('a_size', a.size())
# # a_size torch.Size([3, 4])
# print('b_size', b.size())
# # b_size torch.Size([3, 4, 8])

# idxs = torch.nonzero(a >11)
# print('idxs_size', idxs.size())
# print('idxs', idxs)
# # idxs_size torch.Size([3, 2])

# # print(b.gather(1, idxs))
# import torch
# f = torch.tensor([0.0, 0.05, 0.1, 0.15, 0.2, 0.25, 0.3, 0.35, 0.4, 0.45, 0.5, 0.55, 0.6, 0.65, 0.7, 0.75, 0.8, 0.85, 0.9, 0.95])
# # Create a torch tensor with 20 1-letter strings
# s = ['a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't']
# d1=torch.Tensor(['a','b','c'
# a=torch.tensor([1.0,-2,15, .1, .01, .061,.06,.07,.08,.09,.1,.11,.12,.13,.14,.15,.16,.17,.18,.19,.2,.21,.22,.23,.24,.25,.26,.27,.28,.29,.3,.31,.32,.33,.34,.35,.36,.37,.38,.39,.4,.41,.42,.43,.44,.45,.46,.47,.48,.49,.5,.51,.52,.53,.54,.55,.56,.57,.58,.59,.6,.61,.62,.63,.64,.65,.66,.67,.68,.69,.7,.71,.72,.73,.74,.75,.76,.77,.78,.79,.8,.81,.82,.83,.84,.85,.86,.87,.88,.89,.9,.91,.92,.93,.94,.95,.96,.97,.98,.99])
# l=torch.gt(a,torch.tensor([.7]))
# h=torch.lt(a,torch.tensor([.8]))
# l&h
# (l&h).tolist()

# anchor_embeddings[:10]
# positive_embeddings[:10]
# positives[:10]
# anchors[:10]# 

In [ ]:
y=torch.arange(0,3)
print(y)
z=torch.Tensor([True,False,True])
print(z)
x=torch.Tensor([True,False,True])
print(x)
x=x==True
print(x)
print(y[x])
f = torch.tensor([0.0, 0.05, 0.1, 0.15, 0.2, 0.25, 0.3, 0.35, 0.4, 0.45, 0.5, 0.55])
# Create a torch tensor with 20 1-letter strings
s = ['a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l']

# print(get_indices(f))

# get_hard_negatives(f,s)

ff=torch.reshape(f,(4,3))
ss= [['a 1', 'b 1', 'c 1'],[ 'd 1', 'e 1', 'f 1'],['g 1', 'h 1', 'i'], ['j', 'k', 'l']]
get_hard_negatives(ff,ss)


tensor([0, 1, 2])
tensor([1., 0., 1.])
tensor([1., 0., 1.])
tensor([ True, False,  True])
tensor([0, 2])


In [ ]:
%%time
# Find the closest 5 sentences of the corpus for each query sentence based on cosine similarity
top_k = min(100, len(positive_embeddings))

#for just 1 query
anchor=anchor_embeddings[0]
# We use cosine-similarity and torch.topk to find the highest 5 scores
similarity_scores = model.similarity(anchor, positive_embeddings)[0]
scores, indices = torch.topk(similarity_scores, k=top_k)

gt90=0
gt80=0
gt70=0
lt70=0
for anchor in anchor_embeddings:
    # We use cosine-similarity and torch.topk to find the highest 5 scores
    similarity_scores = model.similarity(anchor, positive_embeddings)[0]
    scores, indices = torch.topk(similarity_scores, k=top_k)
    if scores[0]>90: 
        gt90+=1 
    elif scores[0]>80: 
        gt80+=1 
    elif scores[0]>70: 
        gt70+=1 
    else: 
        lt70+=1
print(f"gt90: {gt90}, gt80: {gt80}, gt70: {gt70}, lt70: {lt70}")
    


gt90: 0, gt80: 0, gt70: 0, lt70: 35258
CPU times: user 14.1 s, sys: 10.3 ms, total: 14.1 s
Wall time: 14.2 s
